# Comparación StrongKalman vs ByteTrack

Ejecuta el detector YOLO **una sola vez** sobre el vídeo de Banyoles, cachea las detecciones por fotograma y luego hace replay a través de `StrongKalmanTrackerV3` y `ByteFieldTracker` por separado. Así la comparación es justa: ambos trackers ven exactamente las mismas detecciones.

**Fragmento analizado**: fotogramas 18 000 – 18 900 (≈ 30 s a 29,97 fps)

## 1 · Imports y configuración

In [1]:
%matplotlib inline
import sys, pickle, cv2, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from tqdm.notebook import tqdm
import ipywidgets as widgets
from IPython.display import display

sys.path.insert(0, '.')
from pipeline_core import (
    preprocess_half, get_video_info, init_undistort_maps,
    pixel_to_field, get_grass_hue, get_player_color, get_torso_crop,
    classify_player_v3, get_hue_signature_masked, select_protos,
    extract_sam_masks_batch, _full_field_dedup, _global_nms,
    StrongKalmanTrackerV3, ByteFieldTracker,
    L_M, A_M, CONF_THRESH,
)
from ultralytics import YOLO

# ── Rutas ──────────────────────────────────────────────────────────────────
VIDEO     = 'data/videos/veo_panoramico_banyoles_1aParte.mp4'
MODEL_PT  = 'runs/detect/modelo_players_v24_panoramic2/weights/best.pt'
HOMO_A    = 'data/calib_alpha1/calib_camA_alpha1_homography.npy'
HOMO_B    = 'data/calib_alpha1/calib_camB_alpha1_homography.npy'
PROTOS_D  = 'prototypes_v3_day.pkl'
PROTOS_N  = 'prototypes_v3_night.pkl'

# ── Parámetros del fragmento ────────────────────────────────────────────────
START_F   = 18_000
N_FRAMES  = 900          # ~30 s
DEVICE    = 'cuda'       # cambiar a 'cpu' si no hay GPU
IMGSZ     = 640

print('Imports OK')

Imports OK


## 2 · Carga de modelo, prototipos y homografías

In [2]:
model = YOLO(MODEL_PT)

with open(PROTOS_D, 'rb') as f:
    protos_day = pickle.load(f)
with open(PROTOS_N, 'rb') as f:
    protos_night = pickle.load(f)

H_a = np.load(HOMO_A)
H_b = np.load(HOMO_B)

info   = get_video_info(VIDEO)
FPS    = info['fps']
VID_W  = info['width']
HALF_H = info['height'] // 2

init_undistort_maps(HALF_H, VID_W)

print(f"FPS={FPS:.3f}  resolución={VID_W}x{info['height']}  frames totales={info['frames']}")

FPS=30.000  resolución=1080x1080  frames totales=82255


## 3 · Caché de detecciones (YOLO se ejecuta una sola vez)

In [3]:
def detect_half(model, half_img, H, protos, conf_thresh, device, imgsz):
    """Devuelve lista de dicts con bbox_px, conf, cls, pos_m, color_desc, hue_sig."""
    results = model.predict(half_img, conf=conf_thresh, verbose=False,
                            device=device, half=(device=='cuda'), imgsz=imgsz)
    dets = []
    if not results or results[0].boxes is None:
        return dets

    boxes_np = results[0].boxes.xyxy.cpu().numpy()
    confs_np = results[0].boxes.conf.cpu().numpy()
    bboxes   = [(int(b[0]), int(b[1]), int(b[2]), int(b[3])) for b in boxes_np]
    # Sin SAM: masks=None para todas las detecciones
    sam_masks = extract_sam_masks_batch(half_img, bboxes, None, device)
    gh = get_grass_hue(half_img) or 55.0

    for idx_b, (box, conf_val) in enumerate(zip(boxes_np, confs_np)):
        x1, y1, x2, y2 = float(box[0]), float(box[1]), float(box[2]), float(box[3])
        crop = half_img[max(0, int(y1)):int(y2), max(0, int(x1)):int(x2)]
        if crop.size == 0:
            continue

        cls_name, _, _, _ = classify_player_v3(crop, protos, grass_hue=gh, mask=None)
        hue_sig           = get_hue_signature_masked(crop, None, gh).tolist()
        _, color_mean, _  = get_player_color(get_torso_crop(crop), gh)

        cx_px = (x1 + x2) / 2.0
        cy_px = float(y2)
        try:
            xm, ym = pixel_to_field(cx_px, cy_px, H)
        except Exception:
            xm, ym = None, None

        dets.append({
            'bbox_px':    (int(x1), int(y1), int(x2), int(y2)),
            'conf':       float(conf_val),
            'cls':        cls_name,
            'pos_m':      (xm, ym) if xm is not None else None,
            'color_desc': color_mean.tolist(),
            'hue_sig':    hue_sig,
        })
    return dets


# ── Bucle principal de detección ─────────────────────────────────────────────
det_cache = {}   # frame_idx → lista de dets unificada (campo completo)
frame_imgs = {}  # frame_idx → frame BGR (para el visualizador)

cap = cv2.VideoCapture(VIDEO)
cap.set(cv2.CAP_PROP_POS_FRAMES, START_F)

for n in tqdm(range(N_FRAMES), desc='Detectando'):
    ret, frame = cap.read()
    if not ret:
        break

    fidx   = START_F + n
    cam_a  = preprocess_half(frame[:HALF_H, :], 'left')
    cam_b  = preprocess_half(frame[HALF_H:,  :], 'right')
    protos = select_protos(frame, protos_day, protos_night)

    dets_a = detect_half(model, cam_a, H_a, protos, CONF_THRESH, DEVICE, IMGSZ)
    dets_b = detect_half(model, cam_b, H_b, protos, CONF_THRESH, DEVICE, IMGSZ)

    unified = _global_nms(
        _full_field_dedup(
            [{**d, 'cam': 'A'} for d in dets_a],
            [{**d, 'cam': 'B'} for d in dets_b],
        ), L_M, A_M)

    det_cache[fidx]  = unified
    frame_imgs[fidx] = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

cap.release()
print(f'Cache construido: {len(det_cache)} frames  |  ejemplo frame {START_F}: {len(det_cache.get(START_F, []))} dets')

Detectando:   0%|          | 0/900 [00:00<?, ?it/s]

Cache construido: 900 frames  |  ejemplo frame 18000: 2 dets


## 4 · Replay a través de ambos trackers

In [4]:
sk_tracker = StrongKalmanTrackerV3(fps=FPS)
bt_tracker = ByteFieldTracker(fps=FPS)

sk_records = []   # [{frame_idx, gid, cls, x, y, speed}]
bt_records = []

for fidx in sorted(det_cache.keys()):
    dets = det_cache[fidx]

    for row in sk_tracker.update(fidx, dets):
        x, y = row['pos_m']
        sk_records.append({'frame': fidx, 'gid': row['gid'],
                            'cls': row['cls'], 'x': x, 'y': y,
                            'speed': row.get('speed_ms', 0.0)})

    for row in bt_tracker.update(fidx, dets):
        x, y = row['pos_m']
        bt_records.append({'frame': fidx, 'gid': row['gid'],
                            'cls': row['cls'], 'x': x, 'y': y,
                            'speed': row.get('speed_ms', 0.0)})

df_sk = pd.DataFrame(sk_records)
df_bt = pd.DataFrame(bt_records)

print(f"StrongKalman → {len(df_sk)} registros  |  IDs únicos: {df_sk['gid'].nunique()}")
print(f"ByteTrack    → {len(df_bt)} registros  |  IDs únicos: {df_bt['gid'].nunique()}")

StrongKalman → 320 registros  |  IDs únicos: 6
ByteTrack    → 318 registros  |  IDs únicos: 12


## 5 · Tabla comparativa de métricas

In [5]:
def tracker_metrics(df, name):
    if df.empty:
        return {}
    n_frames   = df['frame'].nunique()
    n_ids      = df['gid'].nunique()
    n_active   = df.groupby('frame')['gid'].nunique()
    # Fragmentación: ratio IDs únicos / media de IDs activos por frame
    frag_ratio = n_ids / n_active.mean() if n_active.mean() > 0 else float('nan')
    # Longitud media de trayectoria (frames)
    traj_len   = df.groupby('gid')['frame'].count().mean()
    # Velocidad media (m/s)
    speed_mean = df['speed'].mean()

    return {
        'Tracker':               name,
        'IDs únicos':            n_ids,
        'Media IDs/frame':       f"{n_active.mean():.1f}",
        'Ratio fragmentación':   f"{frag_ratio:.2f}",
        'Long. media trayect.':  f"{traj_len:.1f} frames",
        'Vel. media (m/s)':      f"{speed_mean:.2f}",
    }

rows = [tracker_metrics(df_sk, 'StrongKalman'), tracker_metrics(df_bt, 'ByteTrack')]
metrics_df = pd.DataFrame(rows).set_index('Tracker')
display(metrics_df)

,IDs únicos,Media IDs/frame,Ratio fragmentación,Long. media trayect.,Vel. media (m/s)
Tracker,,,,,
StrongKalman,6,1.1,5.31,53.3 frames,4.38
ByteTrack,12,1.1,10.72,26.5 frames,3.33


## 6 · Histograma de IDs únicos por clase

In [6]:
CLASS_ORDER = ['player_home', 'player_away', 'gk_home', 'gk_away', 'referee', 'unknown']
CLS_COLOR   = {'player_home': '#1f77b4', 'player_away': '#d62728',
               'gk_home': '#aec7e8', 'gk_away': '#ffbb78',
               'referee': '#2ca02c', 'unknown': '#999999'}

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for ax, (df, title) in zip(axes, [(df_sk, 'StrongKalman'), (df_bt, 'ByteTrack')]):
    if df.empty:
        ax.set_title(title); continue
    counts = (df.groupby(['cls', 'gid']).size()
                .reset_index().groupby('cls')['gid'].nunique())
    xs = [c for c in CLASS_ORDER if c in counts.index]
    ys = [counts[c] for c in xs]
    bars = ax.bar(xs, ys, color=[CLS_COLOR.get(c,'#cccccc') for c in xs])
    ax.bar_label(bars)
    ax.set_title(f'{title} — IDs únicos por clase', fontweight='bold')
    ax.set_ylabel('IDs únicos')
    ax.tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

C:\Users\xavia\AppData\Local\Temp\ipykernel_17476\895998436.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7 · Diagrama de vida de IDs

In [7]:
def plot_id_lifetime(df, title, ax, max_ids=50):
    if df.empty:
        ax.set_title(title); return
    lifespan = (df.groupby('gid')['frame']
                  .agg(['min','max','count'])
                  .sort_values('min'))
    # Coge el majority class por gid para colorear
    majority_cls = df.groupby('gid')['cls'].agg(lambda s: s.mode()[0])

    show = lifespan.head(max_ids)
    for i, (gid, row) in enumerate(show.iterrows()):
        cls = majority_cls.get(gid, 'unknown')
        col = CLS_COLOR.get(cls, '#999999')
        ax.barh(i, row['max'] - row['min'], left=row['min'], height=0.7,
                color=col, alpha=0.8)
    ax.set_xlabel('Fotograma')
    ax.set_ylabel('ID (orden de aparición)')
    ax.set_title(f'{title} — vida de IDs ({len(lifespan)} total)', fontweight='bold')

    legend_patches = [mpatches.Patch(color=CLS_COLOR[c], label=c)
                      for c in CLASS_ORDER if c in majority_cls.values]
    ax.legend(handles=legend_patches, loc='lower right', fontsize=7)


fig, axes = plt.subplots(1, 2, figsize=(16, 6))
plot_id_lifetime(df_sk, 'StrongKalman', axes[0])
plot_id_lifetime(df_bt, 'ByteTrack',    axes[1])
plt.tight_layout()
plt.show()

C:\Users\xavia\AppData\Local\Temp\ipykernel_17476\1334154089.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8 · Visualizador interactivo fotograma a fotograma

In [8]:
FIELD_IMG_W = 600   # px ancho del canvas del campo
FIELD_IMG_H = int(FIELD_IMG_W * A_M / L_M)

def draw_field_canvas():
    """Canvas verde con líneas básicas."""
    img = np.full((FIELD_IMG_H, FIELD_IMG_W, 3), (34, 139, 34), dtype=np.uint8)
    def fx(xm): return int(xm / L_M * FIELD_IMG_W)
    def fy(ym): return int(ym / A_M * FIELD_IMG_H)
    white = (255, 255, 255)
    # Borde
    cv2.rectangle(img, (fx(0), fy(0)), (fx(L_M), fy(A_M)), white, 2)
    # Línea central
    cv2.line(img, (fx(L_M/2), fy(0)), (fx(L_M/2), fy(A_M)), white, 1)
    # Círculo central
    cy_px = FIELD_IMG_H // 2
    r_px  = int(9.15 / L_M * FIELD_IMG_W)
    cv2.circle(img, (FIELD_IMG_W//2, cy_px), r_px, white, 1)
    return img


CLS_CV2 = {
    'player_home': (31, 119, 180),
    'player_away': (44, 44, 214),
    'gk_home':     (174, 199, 232),
    'gk_away':     (78, 187, 255),
    'referee':     (44, 160, 44),
    'unknown':     (153, 153, 153),
}

def draw_frame(fidx, df, label):
    canvas = draw_field_canvas()
    sub    = df[df['frame'] == fidx]
    for _, row in sub.iterrows():
        px = int(row['x'] / L_M * FIELD_IMG_W)
        py = int(row['y'] / A_M * FIELD_IMG_H)
        px = max(4, min(FIELD_IMG_W-4, px))
        py = max(4, min(FIELD_IMG_H-4, py))
        col = CLS_CV2.get(row['cls'], (153,153,153))
        cv2.circle(canvas, (px, py), 6, col, -1)
        cv2.putText(canvas, str(row['gid']), (px+7, py+4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.35, (255,255,255), 1)
    n_active = len(sub)
    cv2.putText(canvas, f"{label}  frame={fidx}  n={n_active}",
                (6, 16), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255,255,255), 1)
    return cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB)


all_frames = sorted(det_cache.keys())
slider = widgets.IntSlider(
    value=all_frames[0], min=all_frames[0], max=all_frames[-1], step=1,
    description='Frame:', continuous_update=False, layout=widgets.Layout(width='95%')
)

out = widgets.Output()

def on_change(change):
    fidx = change['new']
    with out:
        out.clear_output(wait=True)

        # Campo: SK (izq) | BT (der)
        img_sk = draw_frame(fidx, df_sk, 'StrongKalman')
        img_bt = draw_frame(fidx, df_bt, 'ByteTrack')
        combined_field = np.concatenate([img_sk, img_bt], axis=1)

        # Vídeo original (reducido)
        vid_frame = frame_imgs.get(fidx)

        fig, axes = plt.subplots(1 + (vid_frame is not None), 1,
                                 figsize=(14, 5 + 4*(vid_frame is not None)))
        if vid_frame is None:
            axes = [axes]

        axes[0].imshow(combined_field)
        axes[0].axis('off')
        axes[0].set_title('StrongKalman  ◀──────────────────────▶  ByteTrack',
                           fontsize=11, fontweight='bold')

        if vid_frame is not None:
            axes[1].imshow(vid_frame)
            axes[1].axis('off')
            axes[1].set_title(f'Vídeo original — frame {fidx}', fontsize=10)

        plt.tight_layout()
        plt.show()

slider.observe(on_change, names='value')
display(widgets.VBox([slider, out]))
on_change({'new': slider.value})

## 9 · Trayectorias superpuestas (fragmento de 150 frames)

In [9]:
TRAJ_F0   = START_F
TRAJ_F1   = START_F + 150

def plot_trajectories(df, title, ax):
    sub = df[(df['frame'] >= TRAJ_F0) & (df['frame'] < TRAJ_F1)]
    for gid, grp in sub.groupby('gid'):
        cls = grp['cls'].mode()[0]
        col = [c/255 for c in CLS_CV2.get(cls, (153,153,153))]
        ax.plot(grp['x'], grp['y'], lw=0.8, alpha=0.7, color=col)
        ax.scatter(grp['x'].iloc[-1], grp['y'].iloc[-1], s=20, color=col, zorder=5)
    ax.set_xlim(0, L_M); ax.set_ylim(0, A_M)
    ax.set_aspect('equal')
    ax.set_facecolor('#228b22')
    ax.set_title(f'{title} — trayectorias frames {TRAJ_F0}–{TRAJ_F1}', fontweight='bold')
    ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')

    n_ids = sub['gid'].nunique()
    ax.text(1, 1, f'{n_ids} IDs activos', color='white', fontsize=9,
            transform=ax.transAxes, ha='right', va='bottom')


fig, axes = plt.subplots(1, 2, figsize=(16, 6))
plot_trajectories(df_sk, 'StrongKalman', axes[0])
plot_trajectories(df_bt, 'ByteTrack',    axes[1])

legend_patches = [mpatches.Patch(color=[c/255 for c in CLS_CV2[cl]], label=cl)
                  for cl in CLASS_ORDER if cl in df_sk['cls'].values or cl in df_bt['cls'].values]
axes[1].legend(handles=legend_patches, loc='upper right', fontsize=8)

plt.tight_layout()
plt.show()

C:\Users\xavia\AppData\Local\Temp\ipykernel_17476\109985102.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
